In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [4]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.5 MB/s eta 0:00:00


In [5]:
import numpy as np
import pickle
from gensim.models import Word2Vec

In [6]:
X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_stance = np.load(f"{save_dir}/y_train_stance.npy")
y_test_stance = np.load(f"{save_dir}/y_test_stance.npy")

In [7]:
embedding_matrix = np.load(
    f"{save_dir}/embedding_matrix.npy"
)

In [8]:
w2v_model = Word2Vec.load(
    f"{save_dir}/word2vec.model"
)

In [9]:
with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

vocab_size = len(word_index) + 2

In [10]:
import joblib

stance_encoder = joblib.load(
    f"{save_dir}/stance_encoder.pkl"
)

In [11]:
print("X_train_pad:", X_train_pad.shape)
print("X_test_pad:", X_test_pad.shape)

print("y_train_stance:", y_train_stance.shape)
print("y_test_stance:", y_test_stance.shape)

print("Embedding Matrix:", embedding_matrix.shape)

print("Vocabulary Size:", len(word_index))

X_train_pad: (1166475, 30)
X_test_pad: (291619, 30)
y_train_stance: (1166475,)
y_test_stance: (291619,)
Embedding Matrix: (81958, 100)
Vocabulary Size: 81956


##model 1 computed class wts

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_stance)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_stance
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(19.825871915153986), np.int64(1): np.float64(4.280186695727794), np.int64(2): np.float64(0.3681985189674438)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = lstm_model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 75s 8ms/step - accuracy: 0.5417 - loss: 0.9080 - val_accuracy: 0.5446 - val_loss: 0.8929
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.5770 - loss: 0.8273 - val_accuracy: 0.6463 - val_loss: 0.7416
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 61s 7ms/step - accuracy: 0.5933 - loss: 0.7934 - val_accuracy: 0.5854 - val_loss: 0.8142
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 81s 7ms/step - accuracy: 0.6041 - loss: 0.7694 - val_accuracy: 0.5517 - val_loss: 0.8346


In [ ]:
test_loss, test_acc = lstm_model.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - accuracy: 0.6491 - loss: 0.7389
Test Accuracy: 0.649134635925293


In [ ]:
import numpy as np

y_pred_probs = lstm_model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 27s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.05      0.67      0.09      4902
 Pro Ukraine       0.31      0.56      0.40     22711
      Unsure       0.97      0.66      0.78    264006

    accuracy                           0.65    291619
   macro avg       0.44      0.63      0.42    291619
weighted avg       0.90      0.65      0.74    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  3301    603    998]
 [  4993  12804   4914]
 [ 63361  27450 173195]]


#model 2 class wts + trainable embeddings

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_stance)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_stance
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(19.825871915153986), np.int64(1): np.float64(4.280186695727794), np.int64(2): np.float64(0.3681985189674438)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_2 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model_2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = lstm_model_2.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 101s 12ms/step - accuracy: 0.5647 - loss: 0.8744 - val_accuracy: 0.6115 - val_loss: 0.8115
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 94s 11ms/step - accuracy: 0.6317 - loss: 0.7362 - val_accuracy: 0.6474 - val_loss: 0.7632


In [ ]:
test_loss, test_acc = lstm_model_2.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - accuracy: 0.6128 - loss: 0.8109
Test Accuracy: 0.6127927303314209


In [ ]:
import numpy as np

y_pred_probs = lstm_model_2.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.05      0.65      0.09      4902
 Pro Ukraine       0.25      0.68      0.36     22711
      Unsure       0.97      0.61      0.75    264006

    accuracy                           0.61    291619
   macro avg       0.42      0.64      0.40    291619
weighted avg       0.90      0.61      0.71    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)
print(cm)

[[  3166    969    767]
 [  3912  15444   3355]
 [ 58288  45626 160092]]


##model 3 with wts= 10,3,1 and trainable=true

In [13]:
class_weights = {
    0: 10,
    1: 3,
    2: 1
}

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_3 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [17]:
lstm_model_3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
history = lstm_model_3.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 105s 12ms/step - accuracy: 0.8873 - loss: 0.8583 - val_accuracy: 0.8804 - val_loss: 0.3685
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 101s 12ms/step - accuracy: 0.8861 - loss: 0.7342 - val_accuracy: 0.8793 - val_loss: 0.3570
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 99s 12ms/step - accuracy: 0.8841 - loss: 0.6515 - val_accuracy: 0.8670 - val_loss: 0.3705
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 99s 12ms/step - accuracy: 0.8897 - loss: 0.5790 - val_accuracy: 0.8620 - val_loss: 0.3705


In [19]:
test_loss, test_acc = lstm_model_3.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 42s 5ms/step - accuracy: 0.8795 - loss: 0.3561
Test Accuracy: 0.8794934749603271


In [20]:
import numpy as np

y_pred_probs = lstm_model_3.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step


In [21]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.13      0.28      0.17      4902
 Pro Ukraine       0.48      0.48      0.48     22711
      Unsure       0.95      0.93      0.94    264006

    accuracy                           0.88    291619
   macro avg       0.52      0.56      0.53    291619
weighted avg       0.90      0.88      0.89    291619



In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  1365    270   3267]
 [  1270  10837  10604]
 [  8204  11527 244275]]


In [ ]:
lstm_model_3.save(f"{save_dir}/lstm_stance.keras")

##model 4 (15,4,1) + trainable = true

In [ ]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_4 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model_4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = lstm_model_4.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 102s 12ms/step - accuracy: 0.8588 - loss: 1.0797 - val_accuracy: 0.8721 - val_loss: 0.4365
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 96s 12ms/step - accuracy: 0.8487 - loss: 0.9233 - val_accuracy: 0.8192 - val_loss: 0.4665


In [ ]:
test_loss, test_acc = lstm_model_4.evaluate(
    X_test_pad,
    y_test_stance,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - accuracy: 0.8723 - loss: 0.4359
Test Accuracy: 0.8722545504570007


In [ ]:
import numpy as np

y_pred_probs = lstm_model_4.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.13      0.24      0.17      4902
 Pro Ukraine       0.42      0.50      0.45     22711
      Unsure       0.95      0.92      0.93    264006

    accuracy                           0.87    291619
   macro avg       0.50      0.55      0.52    291619
weighted avg       0.89      0.87      0.88    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  1158    349   3395]
 [   772  11269  10670]
 [  6728  15339 241939]]
